##### Setup

In [ ]:
!pip install transformer_lens

In [ ]:
import math
import os
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable
import datasets
import einops
import numpy as np
import torch
import torch.nn as nn
import wandb
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from torch import Tensor
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer
from transformer_lens.utils import gelu_new, tokenize_and_concatenate
from transformers import GPT2TokenizerFast
MAIN = __name__ == "__main__"

In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

### Playing with GPT-2 small model

In [ ]:
smallGPT2 = HookedTransformer.from_pretrained(
    "gpt2-small",
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,  
)

In [5]:
text = "In a shocking finding, scientists discovered that drinking more water and getting enough sleep can improve your overall health"
tokens = smallGPT2.to_tokens(text).to(device)

In [6]:
tokens

tensor([[50256,   818,   257, 14702,  4917,    11,  5519,  5071,   326,  7722,
           517,  1660,   290,  1972,  1576,  3993,   460,  2987,   534,  4045,
          1535]])

In [7]:
tokens.shape

torch.Size([1, 21])

In [8]:
smallGPT2.to_str_tokens(tokens)

['<|endoftext|>',
 'In',
 ' a',
 ' shocking',
 ' finding',
 ',',
 ' scientists',
 ' discovered',
 ' that',
 ' drinking',
 ' more',
 ' water',
 ' and',
 ' getting',
 ' enough',
 ' sleep',
 ' can',
 ' improve',
 ' your',
 ' overall',
 ' health']

In [9]:
logits, cache = smallGPT2.run_with_cache(tokens)

In [10]:
logits.shape

torch.Size([1, 21, 50257])

In [11]:
probablities = logits.softmax(dim=-1)

In [12]:
nextTokens = smallGPT2.tokenizer.batch_decode(logits.argmax(dim=-1)[0])

Tokens predicted at each position

In [13]:
smallGPT2.to_str_tokens(smallGPT2.to_tokens(nextTokens))

['<|endoftext|>',
 '\n',
 ' the',
 ' statement',
 ' development',
 ',',
 ' the',
 ' have',
 ' that',
 ' the',
 ' water',
 ' than',
 ' is',
 ' eating',
 ' more',
 ' exercise',
 ' can',
 ' help',
 ' your',
 ' health',
 ' health',
 '.']

### Circuits visualisation

In [14]:
!pip install -q circuitsvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.0 MB/s eta 0:00:0000:0100:01


In [15]:
import circuitsvis as cv
from IPython.display import display
from pathlib import Path

vis = cv.attention.attention_patterns(
    tokens=smallGPT2.to_str_tokens(text),
    attention=cache["pattern", 0][0]
)
display(vis)
html_path = Path("attention_patterns.html")
html_path.write_text(vis._repr_html_(), encoding="utf-8")
print(f"Saved to: {html_path.resolve()}")

Saved to: /content/attention_patterns.html


In [16]:
vis_heads = cv.attention.attention_heads(
    tokens=smallGPT2.to_str_tokens(text),
    attention=cache["pattern", 0][0]
)

display(vis_heads)

html_path = Path("attention_heads.html")
html_path.write_text(vis_heads._repr_html_(), encoding="utf-8")

print(f"Saved to: {html_path.resolve()}")

Saved to: /content/attention_heads.html


### Clean Transformer Implementation

In [17]:
@dataclass
class Config:
    dModel: int = 768
    dVocab: int = 50257
    initRange: float = 0.02
    maxContext: int = 1024
    dHead: int = 64
    dMlp: int = 3072
    nHeads: int = 12
    nLayers: int = 12
    debug: bool = True
    eps: float = 1e-5

cfg = Config()

### Layer Normalisation

In [18]:
class LayerNorm(nn.Module):
     def __init__(self, cfg: Config):
          super().__init__()
          self.cfg = cfg 
          self.gamma = nn.Parameter(torch.ones(cfg.dModel))
          self.beta = nn.Parameter(torch.zeros(cfg.dModel))
     
     def forward(self, residual: Tensor) -> Tensor:
          layerMean = residual.mean(dim=-1, keepdim=True)
          layerStd = (
               residual.var(dim = -1, keepdim = True, unbiased=False)
               + self.cfg.eps
               ).sqrt()
          residual = (residual - layerMean) / layerStd
          return residual * self.gamma + self.beta

### Embed

In [19]:
class Embed(nn.Module):
     def __init__(self, cfg: Config):
          super().__init__()
          self.cfg = cfg
          self.wE = nn.Parameter(torch.empty((cfg.dVocab, cfg.dModel)))
          nn.init.normal_(self.wE, std=self.cfg.initRange)
     
     def forward(self, tokens: Tensor) -> Tensor:
          return self.wE[tokens]

In [20]:
class PosEmbed(nn.Module):
     def __init__(self, cfg: Config):
          super().__init__()
          self.cfg = cfg
          self.wPos = nn.Parameter(torch.empty((cfg.maxContext, cfg.dModel)))
          nn.init.normal_(self.wPos, std=self.cfg.initRange)
     
     def forward(self, tokens: Tensor) -> Tensor:
          batch, seqLen = tokens.shape
          return einops.repeat(
               self.wPos[:seqLen],
               "tokenPositions dModel -> batch tokenPositions dModel",
               batch = batch
          )

### Attention mechanism

In [21]:
class Attention(nn.Module):
     IGNORE: Tensor
     def __init__(self, cfg: Config):
          super().__init__()
          self.cfg = cfg

          self.wQ = nn.Parameter(torch.empty((cfg.nHeads, cfg.dModel, cfg.dHead)))
          self.wK = nn.Parameter(torch.empty((cfg.nHeads, cfg.dModel, cfg.dHead)))
          self.wV = nn.Parameter(torch.empty((cfg.nHeads, cfg.dModel, cfg.dHead)))
          self.wO = nn.Parameter(torch.empty((cfg.nHeads, cfg.dHead, cfg.dModel)))

          self.bQ = nn.Parameter(torch.zeros((cfg.nHeads, cfg.dHead)))
          self.bK = nn.Parameter(torch.zeros((cfg.nHeads, cfg.dHead)))
          self.bV = nn.Parameter(torch.zeros((cfg.nHeads, cfg.dHead)))
          self.bO = nn.Parameter(torch.zeros((cfg.dModel,)))

          nn.init.normal_(self.wQ, std=self.cfg.initRange)
          nn.init.normal_(self.wK, std=self.cfg.initRange)
          nn.init.normal_(self.wV, std=self.cfg.initRange)
          nn.init.normal_(self.wO, std=self.cfg.initRange)

          self.register_buffer(
               "IGNORE",
               torch.tensor(float("-inf"), dtype=torch.float32)
          )

     def forward(self, residual: Tensor) -> Tensor:
          query = residual.unsqueeze(1) @ self.wQ.unsqueeze(0)
          key = residual.unsqueeze(1) @ self.wK.unsqueeze(0)
          value = residual.unsqueeze(1) @ self.wV.unsqueeze(0)

          query = query.permute(0, 2, 1, 3) + self.bQ
          key = key.permute(0, 2, 1, 3) + self.bK
          value = value.permute(0, 2, 1, 3) + self.bV

          attentionScores = (
               query.permute(0, 2, 1, 3)
               @
               key.permute(0, 2, 1, 3).transpose(-1, -2)
          )
          attentionScores = attentionScores / (self.cfg.dHead ** 0.5)
          maskedScores = self.applyMask(attentionScores)
          attentionPatterns = maskedScores.softmax(dim=-1)

          valueMul = (
               attentionPatterns
               @
               value.permute(0, 2, 1, 3)
          )
          attentionOutPerHead = valueMul @ self.wO.unsqueeze(0)
          attentionOutput = attentionOutPerHead.sum(dim = 1)
          attentionOutput = attentionOutput + self.bO
          return attentionOutput

     def applyMask(self, attentionScores: Tensor) -> Tensor:
          maskMatrix = torch.ones(
               attentionScores.size(-2),
               attentionScores.size(-1),
               device = attentionScores.device
          )
          mask = torch.triu(
               maskMatrix,
               diagonal = 1
          ).bool()
          attentionScores = attentionScores.masked_fill_(mask, self.IGNORE)
          return attentionScores

### MLP

In [22]:
class MLP(nn.Module):
     def __init__(self, cfg: Config):
          super().__init__()
          self.cfg = cfg

          self.hiddenLayerW = nn.Parameter(torch.empty((cfg.dModel, cfg.dMlp)))
          self.hiddenLayerB = nn.Parameter(torch.zeros((cfg.dMlp,)))

          self.outputW = nn.Parameter(torch.empty((cfg.dMlp, cfg.dModel)))
          self.outputB = nn.Parameter(torch.zeros((cfg.dModel,)))

          nn.init.normal_(self.hiddenLayerW, std=self.cfg.initRange)
          nn.init.normal_(self.outputW, std=self.cfg.initRange)

     def forward(self, residual:Tensor) -> Tensor:
          hiddenLayer = residual @ self.hiddenLayerW + self.hiddenLayerB
          actFunc = torch.nn.functional.gelu(hiddenLayer, approximate="tanh")
          outLayer = actFunc @ self.outputW + self.outputB
          return outLayer

### Transformer Block

x

↓

LayerNorm

↓

Masked Self-Attention

↓

Residual Add

↓

LayerNorm

↓

MLP

↓

Residual Add

↓

x

In [23]:
class TransformerBlock(nn.Module):
     def __init__(self, cfg: Config):
          super().__init__()
          self.cfg = cfg
          self.layerNorm1 = LayerNorm(cfg)
          self.attention = Attention(cfg)
          self.layerNorm2 = LayerNorm(cfg)
          self.mlp = MLP(cfg)

     def forward(self, x: Tensor) -> Tensor:
          ln1 = self.layerNorm1(x)
          maskedSelfAttention = self.attention(ln1)
          residualConn1 = x + maskedSelfAttention
          ln2 = self.layerNorm2(residualConn1)
          mlpOutput = self.mlp(ln2)
          residualConn2 = mlpOutput + residualConn1

          finalOutput = residualConn2
          return finalOutput

### Unembed

In [24]:
class Unembed(nn.Module):
     def __init__(self, cfg: Config):
          super().__init__()
          self.cfg = cfg

          self.w = nn.Parameter(torch.empty((cfg.dModel, cfg.dVocab)))
          nn.init.normal_(self.w, std=self.cfg.initRange)

          self.b = nn.Parameter(torch.zeros((cfg.dVocab,)), requires_grad=False)

     def forward(self, finalOutput: Tensor) -> Tensor:
          logits = finalOutput @ self.w + self.b
          return logits

### Demo transformer

In [25]:
class MyTransformer(nn.Module):
     def __init__(self, cfg: Config):
          super().__init__()
          self.cfg = cfg
          self.embed = Embed(cfg)
          self.posEmbed = PosEmbed(cfg)
          self.blocks = nn.ModuleList(
               [
                    TransformerBlock(cfg)
                    for _ in range(cfg.nLayers)
                ]
          )
          self.ln = LayerNorm(cfg)
          self.unembed = Unembed(cfg)

     def forward(self, tokens: Tensor) -> Tensor:
          residual = self.embed(tokens) + self.posEmbed(tokens)
          for block in self.blocks:
               residual = block(residual)
          normResidual = self.ln(residual)
          logits = self.unembed(normResidual)
          return logits

### Training

#### Transformer model

In [26]:
modelConfig = Config(
     dModel = 32,
     dVocab = smallGPT2.cfg.d_vocab,
     maxContext = 128,
     nHeads = 16,
     dHead = 2,
     dMlp = 32 * 4,
     nLayers = 4,
     debug = False
)

In [27]:
model = MyTransformer(modelConfig)

### Training arguments

In [28]:
@dataclass
class TrainingArgs:
     batchSize: int = 32
     epochs: int = 10
     maxEpochSteps: int = 500
     lr: float = 1e-3    
     weightDecay: float = 1e-2
     evalPrompt: str = "Once upon a time"

In [29]:
args = TrainingArgs()

### Data

In [30]:
dataset = datasets.load_dataset("roneneldan/TinyStories", split="train")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [31]:
dataset.shape

(2119719, 1)

In [32]:
dataset[0]["text"]

'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'

In [33]:
tokenized_dataset = tokenize_and_concatenate(
    dataset.select(range(5000)),
    smallGPT2.tokenizer,
    streaming=False,
    max_length=model.cfg.maxContext,
    column_name="text",
    add_bos_token=True,
    num_proc=8,
)

Map (num_proc=8):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [34]:
tokenized_dataset.shape

(8130, 1)

In [35]:
datasetTrainTest = tokenized_dataset.train_test_split(
     test_size = 1000
)

trainLoader = DataLoader(
     datasetTrainTest["train"],
     batch_size = args.batchSize,
     shuffle = True
)

testLoader = DataLoader(
     datasetTrainTest["test"],
     batch_size = args.batchSize,
     shuffle = False
)

In [36]:
firstBatch = trainLoader.dataset[: args.batchSize]
firstBatch["tokens"].shape

torch.Size([32, 128])

### Training loop

In [37]:
class TransformerTrainer:
     def __init__(self, args: TrainingArgs, model: MyTransformer, tokenizer):
          self.model = model
          self.args = args
          self.tokenizer = tokenizer
          self.optimizer = torch.optim.AdamW(
               self.model.parameters(), 
               lr = args.lr,
               weight_decay = args.weightDecay
          )
          self.step = 0
          self.trainLoader = trainLoader
          self.testLoader = testLoader

     def logProbablity(self, logits, tokens):
          logProbs = logits.log_softmax(dim=-1)
          logProbs = logProbs[:, :-1, :]
          nextToken = tokens[:, 1:]
          logProbs = logProbs.gather(
               dim=-1,
               index=nextToken.unsqueeze(-1)
          ).squeeze(-1)
          return logProbs

     def trainingStep(self, batch):
          self.model.train()
          tokens = batch["tokens"].to(device)
          logits = self.model(tokens)   # batch seq, vocabSize
          loss = -self.logProbablity(logits, tokens).mean()
          self.optimizer.zero_grad()
          loss.backward()
          self.optimizer.step()
          self.step += 1
          return loss.item()
     
     @torch.inference_mode()
     def evaluate(self):
          self.model.eval()
          totalSamples = 0
          totalCorrect = 0
          for batch in tqdm(self.testLoader, desc="Evaluating"):
               tokens = batch["tokens"].to(device)
               logits = self.model(tokens)
               logits = logits[:, :-1, :]
               predictedTokens = logits.argmax(dim=-1)
               targetTokens = tokens[:, 1:]
               totalCorrect += (predictedTokens == targetTokens).sum().item()
               totalSamples += targetTokens.numel()
          accuracy = totalCorrect / totalSamples
          self.model.train()
          return accuracy
     
     @torch.inference_mode()
     def sample(self, prompt, maxTokensGenerated=50):
          self.model.eval()
          tokenIds = self.tokenizer.encode(prompt)
          tokens = torch.tensor(tokenIds, dtype=torch.long, device=device).unsqueeze(0)
          for _ in range(maxTokensGenerated):
               tokensForModel = tokens[:, -self.model.cfg.maxContext:]
               logits = self.model(tokensForModel)
               nextTokenLogits = logits[:, -1, :]
               nextToken = nextTokenLogits.argmax(dim = -1, keepdim = True)
               tokens = torch.cat([tokens, nextToken], dim=1)
          text = self.tokenizer.decode(tokens[0].tolist())
          self.model.train()
          return text
     
     def train(self):
          print("Sample before training")
          print(self.sample(self.args.evalPrompt, maxTokensGenerated=50))
          latestAccuracy = float("nan")

          for epoch in range(self.args.epochs):
               print(f"Epoch {epoch + 1}/{self.args.epochs}")
               running_loss = 0.0
               progress_bar = tqdm(
                    self.trainLoader,
                    total=self.args.maxEpochSteps,
                    desc="Training",
               )
               for step_in_epoch, batch in enumerate(progress_bar, start=1):
                    if step_in_epoch > self.args.maxEpochSteps:
                         break
                    loss = self.trainingStep(batch)
                    running_loss += loss
                    avg_loss = running_loss / step_in_epoch
                    progress_bar.set_postfix(
                         loss=f"{loss:.3f}",
                         avg_loss=f"{avg_loss:.3f}",
                         accuracy=f"{latestAccuracy:.3f}",
                    )
               latestAccuracy = self.evaluate()
               print(f"Epoch {epoch + 1} finished")
               print(f"Average train loss: {avg_loss:.4f}")
               print(f"Test accuracy: {latestAccuracy:.4f}")
               print("Sample after epoch:")
               print(self.sample(self.args.evalPrompt, maxTokensGenerated=50))
               print()

In [38]:
trainer = TransformerTrainer(
     args=args,
     model=model,
     tokenizer=smallGPT2.tokenizer
)

In [ ]:
trainer.train()